## 🔰PyTorchでニューラルネットワーク基礎 #25 【トークナイズ編・BPE】

### 内容
* Qiitaの記事と連動しています

### データについて
cc100から３ヶ国語抽出。それぞれ10万行。text列に対応する言語の文書
* 英語：cc100_en_100k.parquet
* 日本語：cc100_ja_100k.parquet
* 韓国語：cc100_ko_100k.parquet

### ちょっとした注意点
* **データやトークナイザーの保存先に注意**。適宜修正してください。
* ByteLevel版のみ。Metaspaceで試す場合はコメントの部分を変更
* 利用ライブラリ：tokenizers
* データ数が多い場合はシャッフルする部分を少し変更しないとNG

## BPE + ByteLevelでの動作確認
* Metaspaceのときはコメントの部分を変更すればOK

In [ ]:
import random
import pandas as pd
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

# ByteLevelとMetaspaceの選択部分
# ByteLevel: 空白のprefixを明示的に使う
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)    # ByteLevelを利用
# tokenizer.pre_tokenizer = pre_tokenizers.Metaspace(replacement="▁")        # Metaspaceを利用

trainer = trainers.BpeTrainer(
    vocab_size=10_000,        # ここで語彙数を指定
    special_tokens=["<pad>", "<bos>", "<eos>", "<unk>", "<mask>"],  # この順番で特殊トークンにIDが割り当てられる
    min_frequency=2
)


# 4. parquetファイルから直接学習
paths = ["./data/cc100_ja_100k.parquet","./data/cc100_en_100k.parquet","./data/cc100_ko_100k.parquet"]

# 文章をランダムに並び替えるパターン
def mixed_iterator(paths):
    texts = []
    for p in paths:
        # text列だけ読み込む
        df = pd.read_parquet(p, columns=["text"])
        texts.extend(df["text"].tolist())
    
    # シャッフル（数百万件程度までならこの方法でOKなはず。あとJupyterLabでもOKなはず。）
    random.shuffle(texts)
    
    for t in texts:
        yield t

# 学習部分
tokenizer.train_from_iterator(mixed_iterator(paths), trainer=trainer)

# 5. トークナイザーを保存
tokenizer.save("./tokenizer/bpe_bytelevel_10k.json")

In [9]:
from tokenizers import Tokenizer

# 保存したトークナイザーで確認
tokenizer = Tokenizer.from_file("tokenizer/bpe_bytelevel_10k.json")

print("特殊トークンID:")
print(f"<pad>: {tokenizer.token_to_id('<pad>')}")
print(f"<bos>: {tokenizer.token_to_id('<bos>')}")
print(f"<eos>: {tokenizer.token_to_id('<eos>')}")
print(f"<unk>: {tokenizer.token_to_id('<unk>')}")
print(f"<mask>: {tokenizer.token_to_id('<mask>')}")
print(f"size: {tokenizer.get_vocab_size()}")

特殊トークンID:
<pad>: 0
<bos>: 1
<eos>: 2
<unk>: 3
<mask>: 4
size: 10000


In [ ]:
# ByteLevelのときは次を利用する。なくても動作するけど、おもしろ表示になるぞ😆
from tokenizers import decoders
tokenizer.decoder = decoders.ByteLevel()

text_list = [
    "これは日本語のテストです🍀",
    "Awesome blog! Do you have any suggestions",
    "📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.",
    "你好",
    "𐀀𐀁"
]
    
for text in text_list:
    encoded = tokenizer.encode(text)
    print(f"文章: {text}")
    print("トークン:", encoded.tokens)
    print("ID:", encoded.ids)
    print(f"デコード: {tokenizer.decode(encoded.ids, skip_special_tokens=False)}\n")

文章: これは日本語のテストです🍀
トークン: ['ĠãģĵãĤĮãģ¯', 'æĹ¥æľ¬', 'èªŀ', 'ãģ®', 'ãĥĨ', 'ãĤ¹ãĥĪ', 'ãģ§ãģĻ', 'ðŁ', 'į', 'Ģ']
ID: [7375, 1478, 2331, 216, 972, 1564, 387, 3558, 183, 170]
デコード:  これは日本語のテストです🍀

文章: Awesome blog! Do you have any suggestions
トークン: ['ĠA', 'w', 'esome', 'Ġblog', '!', 'ĠDo', 'Ġyou', 'Ġhave', 'Ġany', 'Ġsuggest', 'ions']
ID: [417, 91, 8305, 2584, 5, 3432, 311, 522, 979, 5103, 1112]
デコード:  Awesome blog! Do you have any suggestions

文章: 📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.
トークン: ['ĠðŁ', 'ĵ', '·', 'ãĢĢ', 'ìłķ', 'êµĲ', 'íķĺë©´ìĦľ', 'ĠìĻĦ', 'ë²½', 'íķĺê²Į', 'ĠìķĦ', 'ëĤł', 'ë¡ľê·¸', 'Ġì¹´', 'ë©ĶëĿ¼', 'ë¥¼', 'Ġìŀ¬', 'íĺĦ', 'íķľ', 'Ġê²ĥ', 'Ġê°Ļëĭ¤', '.']
ID: [6328, 189, 119, 1379, 505, 1060, 4457, 2858, 5261, 1492, 529, 2532, 2133, 2224, 8773, 360, 1605, 1888, 308, 574, 5709, 18]
デコード:  📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.

文章: 你好
トークン: ['Ġ', 'ä½', 'ł', 'å¥½']
ID: [169, 419, 202, 2298]
デコード:  你好

文章: 𐀀𐀁
トークン: ['Ġ', 'ð', 'Ĳ', 'Ģ', 'Ģ', 'ð', 'Ĳ', 'Ģ', 'ģ']
ID: [169, 162, 186, 170, 170, 162, 186, 170

### HuggingFaceのAutoTokenizerで利用できるように保存する

In [ ]:
from transformers import PreTrainedTokenizerFast
from tokenizers import decoders

# まず今のjsonをラップ
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="tokenizer/bpe_bytelevel_10k.json",
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>",
    mask_token="<mask>",
    add_prefix_space=True
)
tokenizer.backend_tokenizer.decoder = decoders.ByteLevel()  # なくても良い。decoder時の文字化け風を避ける
# HF形式で保存
tokenizer.save_pretrained("hf_type_tokenizer")

('hf_type_tokenizer/tokenizer_config.json',
 'hf_type_tokenizer/special_tokens_map.json',
 'hf_type_tokenizer/tokenizer.json')

In [13]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("hf_type_tokenizer")


text_list = [
    "これは日本語のテストです🍀",
    "Awesome blog! Do you have any suggestions",
    "📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.",
    "你好",
    "𐀀𐀁"
]
    
for text in text_list:
    encoded = tokenizer(text)
    print(f"文章: {text}")
    print("トークン:", encoded)
    tokens = [tokenizer.decode(id) for id in encoded["input_ids"]]
    print(f"分割: {tokens}")
    print("ID:", encoded["input_ids"])
    print(f"デコード: {tokenizer.decode(encoded['input_ids'])}\n")

文章: これは日本語のテストです🍀
トークン: {'input_ids': [7375, 1478, 2331, 216, 972, 1564, 387, 3558, 183, 170], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
分割: [' これは', '日本', '語', 'の', 'テ', 'スト', 'です', '�', '�', '�']
ID: [7375, 1478, 2331, 216, 972, 1564, 387, 3558, 183, 170]
デコード:  これは日本語のテストです🍀

文章: Awesome blog! Do you have any suggestions
トークン: {'input_ids': [417, 91, 8305, 2584, 5, 3432, 311, 522, 979, 5103, 1112], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
分割: [' A', 'w', 'esome', ' blog', '!', ' Do', ' you', ' have', ' any', ' suggest', 'ions']
ID: [417, 91, 8305, 2584, 5, 3432, 311, 522, 979, 5103, 1112]
デコード:  Awesome blog! Do you have any suggestions

文章: 📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.
トークン: {'input_ids': [6328, 189, 119, 1379, 505, 1060, 4457, 2858, 5261, 1492, 529, 2532, 2133, 2224, 8773, 360, 1605, 1888, 308, 574, 5709, 18], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [27]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("hf_type_tokenizer")

text_list = [
    "これは日本語のテストです🍀",
    "Awesome blog! Do you have any suggestions",
    "📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.",
    "你好",
    "𐀀𐀁"
]
    
for text in text_list:
    encoded = tokenizer(text)
    print(f"文章: {text}")
    print("トークン:", encoded)
    tokens = [tokenizer.decode(id) for id in encoded["input_ids"]]
    print(f"分割: {tokens}")
    print(f"ID: {encoded['input_ids']}")
    print(f"デコード: {tokenizer.decode(encoded['input_ids'])}\n")

文章: これは日本語のテストです🍀
トークン: {'input_ids': [7375, 1478, 2331, 216, 972, 1564, 387, 3558, 183, 170], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
分割: [' これは', '日本', '語', 'の', 'テ', 'スト', 'です', '�', '�', '�']
ID: [7375, 1478, 2331, 216, 972, 1564, 387, 3558, 183, 170]
デコード:  これは日本語のテストです🍀

文章: Awesome blog! Do you have any suggestions
トークン: {'input_ids': [417, 91, 8305, 2584, 5, 3432, 311, 522, 979, 5103, 1112], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
分割: [' A', 'w', 'esome', ' blog', '!', ' Do', ' you', ' have', ' any', ' suggest', 'ions']
ID: [417, 91, 8305, 2584, 5, 3432, 311, 522, 979, 5103, 1112]
デコード:  Awesome blog! Do you have any suggestions

文章: 📷　정교하면서 완벽하게 아날로그 카메라를 재현한 것 같다.
トークン: {'input_ids': [6328, 189, 119, 1379, 505, 1060, 4457, 2858, 5261, 1492, 529, 2532, 2133, 2224, 8773, 360, 1605, 1888, 308, 574, 5709, 18], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 